<a href="https://colab.research.google.com/github/TomonoriGH/colab_git/blob/main/arbit_nn.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

### 1. 設定とモデルの定義
入力層と出力層のサイズを指定し、シンプルなニューラルネットワークを定義します。

In [1]:
!pip install torchinfo

In [7]:
torch.set_default_device('cuda')

#### example

In [2]:
import torch
import torch.nn as nn
import torch.optim as optim
import numpy as np

# --- パラメータ設定 ---
INPUT_SIZE = 10   # 入力層の大きさ
OUTPUT_SIZE = 2   # 出力層の大きさ (例: 2クラス分類)
HIDDEN_SIZE = 64  # 隠れ層の大きさ
BATCH_SIZE = 16
LEARNING_RATE = 0.01
EPOCHS = 20

# モデルの定義
class SimpleNet(nn.Module):
    def __init__(self, input_size, hidden_size, output_size):
        super(SimpleNet, self).__init__()
        self.fc1 = nn.Linear(input_size, hidden_size)
        self.relu = nn.ReLU()
        self.fc2 = nn.Linear(hidden_size, output_size)

    def forward(self, x):
        x = self.fc1(x)
        x = self.relu(x)
        x = self.fc2(x)
        return x

model = SimpleNet(INPUT_SIZE, HIDDEN_SIZE, OUTPUT_SIZE)
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=LEARNING_RATE)
print("Model initialized.")

Model initialized.


#### selfmade

In [44]:
import torch
import torch.nn as nn
import torch.optim as optim
import numpy as np
from torchinfo import summary

# --- パラメータ設定 ---
INPUT_SIZE = 6   # 入力層の大きさ
OUTPUT_SIZE = 6   # 出力層の大きさ (例: 2クラス分類)
NUM_SAMPLE = 10000
HIDDEN_SIZE = 64  # 隠れ層の大きさ
HIDDEN_SIZE = 6  # 隠れ層の大きさ
BATCH_SIZE = 16
LEARNING_RATE = 0.01
EPOCHS = 200
LAYER_LENGTH = 20

model = nn.Sequential(*[
        nn.Linear(
              INPUT_SIZE if i == 0 else HIDDEN_SIZE,
              OUTPUT_SIZE if i == LAYER_LENGTH - 1 else HIDDEN_SIZE
        )
      if ii == 0 else
      nn.ReLU()
      for i in range(LAYER_LENGTH)
      for ii in range(1 if i == LAYER_LENGTH - 1 else 2)
    ])
criterion = nn.CrossEntropyLoss()
criterion = nn.MSELoss()
optimizer = optim.Adam(model.parameters(), lr=LEARNING_RATE)
print("Model initialized.")
summary(model)

Model initialized.


Layer (type:depth-idx)                   Param #
Sequential                               --
├─Linear: 1-1                            42
├─Linear: 1-2                            42
Total params: 84
Trainable params: 84
Non-trainable params: 0

### 2. ミニバッチ生成関数とダミーデータ
学習に使用するサンプルデータと、バッチを取得する関数を用意します。

#### example

In [ ]:
# ダミーデータの作成 (入力: 100サンプル, 正解ラベル: 0 or 1)
X_train = torch.randn(100, INPUT_SIZE)
y_train = torch.randint(0, OUTPUT_SIZE, (100,))

def get_mini_batches(X, y, batch_size):
    indices = np.arange(len(X))
    np.random.shuffle(indices)
    for i in range(0, len(X), batch_size):
        batch_idx = indices[i:i + batch_size]
        yield X[batch_idx], y[batch_idx]

print("Data and batch generator ready.")

#### selfmade

In [45]:
def ylinside(l) :
  l = [str(e) for e in l]
  v1 = int("".join(l[:3])) * int("".join(l[3:]))
  return [int(e) for e in list(str(v1).zfill(6))]

xl,yl,X_train,y_train = [None] * 4

def gen_train_data() :
  global xl,yl,X_train,y_train
  xl = np.random.randint(0,10,size=(NUM_SAMPLE,INPUT_SIZE)).tolist()
  yl = [ylinside(xl[i]) for i in range(NUM_SAMPLE)]

  X_train = torch.tensor(xl,dtype=torch.float)
  y_train = torch.tensor(yl,dtype=torch.float)

def get_mini_batches(X, y, batch_size):
    gen_train_data()
    indices = np.arange(len(X))
    np.random.shuffle(indices)
    for i in range(0, len(X), batch_size):
        batch_idx = indices[i:i + batch_size]
        yield X[batch_idx], y[batch_idx]


print("Data and batch generator ready.")

Data and batch generator ready.


### 3. 学習実行 (ボタン一つで実行相当)
このセルを実行することで学習が始まります。

In [ ]:
list(get_mini_batches(X_train,y_train,1))

In [47]:
def train():
    model.train()
    for epoch in range(EPOCHS):
        total_loss = 0
        for batch_X, batch_y in get_mini_batches(X_train, y_train, BATCH_SIZE):
            optimizer.zero_grad()
            outputs = model(batch_X)
            loss = criterion(outputs, batch_y)
            loss.backward()
            optimizer.step()
            total_loss += loss.item()

        if (epoch + 1) % 5 == 0:
            print(f"Epoch [{epoch+1}/{EPOCHS}], Loss: {total_loss:.4f}")
    print("Training completed.")

train()

Epoch [5/200], Loss: 4400.1083
Epoch [10/200], Loss: 4386.0340
Epoch [15/200], Loss: 4376.5912
Epoch [20/200], Loss: 4391.7210
Epoch [25/200], Loss: 4387.6223
Epoch [30/200], Loss: 4358.6364
Epoch [35/200], Loss: 4362.4947
Epoch [40/200], Loss: 4377.3625
Epoch [45/200], Loss: 4385.0383
Epoch [50/200], Loss: 4379.0747
Epoch [55/200], Loss: 4355.6205
Epoch [60/200], Loss: 4368.8291
Epoch [65/200], Loss: 4347.3585
Epoch [70/200], Loss: 4363.1905
Epoch [75/200], Loss: 4358.6198
Epoch [80/200], Loss: 4387.7408
Epoch [85/200], Loss: 4391.9634
Epoch [90/200], Loss: 4400.6148
Epoch [95/200], Loss: 4384.1771
Epoch [100/200], Loss: 4382.7900
Epoch [105/200], Loss: 4380.0147
Epoch [110/200], Loss: 4369.4139
Epoch [115/200], Loss: 4395.3390
Epoch [120/200], Loss: 4388.3929
Epoch [125/200], Loss: 4373.5823
Epoch [130/200], Loss: 4386.8441
Epoch [135/200], Loss: 4381.5295
Epoch [140/200], Loss: 4383.6656
Epoch [145/200], Loss: 4392.9683
Epoch [150/200], Loss: 4378.3197
Epoch [155/200], Loss: 4336.66

### 4. 使用・評価 (ボタン一つで実行相当)
学習済みモデルを使用して推論を行う例です。

#### example

In [12]:
def evaluate():
    model.eval()
    # 新しい未知のデータを作成
    test_input = torch.randn(5, INPUT_SIZE)

    with torch.no_grad():
        predictions = model(test_input)
        # 最大値のインデックスを取得 (クラス予測)
        _, predicted_classes = torch.max(predictions, 1)

    print("Input Data Shape:", test_input.shape)
    print("Predicted Classes:", predicted_classes.numpy())

evaluate()

Input Data Shape: torch.Size([5, 6])
Predicted Classes: [0 0 0 0 0]


#### selfmade

In [68]:
def evaluate():
    model.eval()
    # 新しい未知のデータを作成
    test_input,answer = next(get_mini_batches(X_train,y_train,100))
    print("input",test_input[0])
    print("answer",answer[0])

    with torch.no_grad():
        predictions = model(test_input)
        print(predictions[0])
        # 最大値のインデックスを取得 (クラス予測)
        _, predicted_classes = torch.max(predictions, 1)

    print("Input Data Shape:", test_input.shape)
    # print("Predicted Classes:", predicted_classes.numpy())

evaluate()

input tensor([1., 4., 4., 1., 7., 5.])
answer tensor([0., 2., 5., 2., 0., 0.])
tensor([-1.2217,  2.9934,  4.3223,  4.0417,  4.5022,  3.7224])
Input Data Shape: torch.Size([100, 6])
